In [1]:
import pandas as pd
import os
import re

In [3]:
# 매장 테이블 ✅ 
매장 = pd.DataFrame(columns=["매장_id", "매장명", "매장_비밀번호"])

# 협력사 테이블 ✅ 
협력사 = pd.DataFrame(columns=["협력사_id", "협력사명"])

# 품목 테이블 ✅ 
품목 = pd.DataFrame(columns=["품목_id", "협력사_id", "품목명", "규격", "단위", "입고단가", "입고단위", "입고단위단가"])

# 매장_재고 테이블  ✅ 
매장_재고 = pd.DataFrame(columns=["매장_id", "품목_id", "기간", "매장_재고량"])

# 매장_발주 테이블  ✅ 
매장_발주 = pd.DataFrame(columns=["매장_id", "품목_id", "기간", "매장_발주량"])

# 창고_입고 테이블 ✅ 
창고_입고 = pd.DataFrame(columns=["매장_id", "품목_id", "기간", "창고_입고량"])

# 창고_출고 테이블 ✅ 
창고_출고 = pd.DataFrame(columns=["매장_id", "품목_id", "기간", "창고_출고량"])

# 창고_재고 테이블 ✅ 
창고_재고 = pd.DataFrame(columns=["매장_id", "품목_id", "기간", "창고_재고량"])

# 창고_발주 테이블  
창고_발주 = pd.DataFrame(columns=["협력사_id", "품목_id", "기간", "창고_발주량"])

In [5]:
# 🔹 앞뒤 공백 제거 & 중간에 여러 개의 공백을 하나의 공백으로 정리하는 함수
def clean_text(text):
    if isinstance(text, str):  # 문자열일 경우만 처리
        return " ".join(text.strip().split())
    return text  # NaN 등 문자열이 아닐 경우 그대로 반환

## 매장

In [7]:
# 매장 데이터 추가
매장_data = [
    ("ST_101", "admin", "rhksfl321!"),
    ("ST_102", "창고", "ckdrh123!"),
    ("ST_103", "푸른솔", "vnfms123!"),
    ("ST_104", "의과대학", "dmlrhk123!"),
    ("ST_105", "중앙도서관", "wnddkd123!"),
    ("ST_106", "학생회관", "gkrtod123!"),
    ("ST_107", "예술디자인대", "elwkdls123!"),
    ("ST_108", "선승관", "tjstmd123!"),
    ("ST_109", "공학관", "rhdgkr123!"),
    ("ST_110", "멀티미디어관", "ajfxl123!"),
    ("ST_111", "제2기숙사", "rltnr123!"),
]

# 데이터프레임에 추가
매장 = pd.concat([매장, pd.DataFrame(매장_data, columns=매장.columns)], ignore_index=True)
매장

,매장_id,매장명,매장_비밀번호
0,ST_101,admin,rhksfl321!
1,ST_102,창고,ckdrh123!
2,ST_103,푸른솔,vnfms123!
3,ST_104,의과대학,dmlrhk123!
4,ST_105,중앙도서관,wnddkd123!
5,ST_106,학생회관,gkrtod123!
6,ST_107,예술디자인대,elwkdls123!
7,ST_108,선승관,tjstmd123!
8,ST_109,공학관,rhdgkr123!
9,ST_110,멀티미디어관,ajfxl123!


## 협력사

In [9]:
# 협력사 데이터
협력사_data = [
    ("CO_101", "비스토리코리아"),
    ("CO_102", "성원애드피아"),
    ("CO_103", "휴럼"),
    ("CO_104", "세미기업"),
    ("CO_105", "손맛커피"),
    ("CO_106", "성유엔터프라이즈"),
]

# 데이터프레임에 추가
협력사 = pd.concat([협력사, pd.DataFrame(협력사_data, columns=협력사.columns)], ignore_index=True)
협력사

,협력사_id,협력사명
0,CO_101,비스토리코리아
1,CO_102,성원애드피아
2,CO_103,휴럼
3,CO_104,세미기업
4,CO_105,손맛커피
5,CO_106,성유엔터프라이즈


## 품목

In [11]:
# 엑셀 파일 읽기
file_path = "data/품목.xlsx"
df = pd.read_excel(file_path, dtype=str)  # 모든 데이터를 문자열로 읽어오기

# 🔹 모든 문자열 컬럼에 적용 (apply()와 map() 조합 사용)
df = df.apply(lambda col: col.map(clean_text) if col.dtype == "O" else col)

# 품목_id 자동 증가 (IT_101부터 시작)
df["품목_id"] = [f"IT_{101 + i}" for i in range(len(df))]

# 컬럼 정렬 (원하는 순서대로)
df = df[["품목_id", "협력사_id", "품목명", "규격", "단위", "입고단가", "입고단위", "입고단위단가"]]

# 기존 품목 테이블과 병합
품목 = pd.concat([품목, df], ignore_index=True)
품목

,품목_id,협력사_id,품목명,규격,단위,입고단가,입고단위,입고단위단가
0,IT_101,CO_101,종이컵(-12oz),1box=1000ea,ea,37,1000,37000
1,IT_102,CO_101,종이컵-16oz,1box=1000ea,ea,43,1000,43000
2,IT_103,CO_101,"종이컵뚜껑 (12,16oz)",1box=1000ea,ea,20,1000,20000
3,IT_104,CO_101,아이스(페트)컵(14oz),1box=1000ea,ea,42,1000,42000
4,IT_105,CO_101,아이스컵(16oz),1box=1000ea,ea,53,1000,53000
...,...,...,...,...,...,...,...,...
66,IT_167,CO_105,디카페인 원두,1box=10kg,ea,19000,10,190000
67,IT_168,CO_106,티칸 캐모마일,1box=10pack,pack,4000,1,4000
68,IT_169,CO_106,티칸 페퍼민트,1box=10pack,pack,4000,1,4000
69,IT_170,CO_106,티칸 루이보스오렌지,1box=10pack,pack,4000,1,4000


## 창고_출고

In [13]:
# 🔹 창고_출고 테이블 초기화
창고_출고 = pd.DataFrame(columns=["매장_id", "품목_id", "기간", "창고_출고량"])

# 🔹 데이터가 저장된 폴더 경로
data_dir = "data/입출고관리대장/"

# 🔹 파일명 패턴: "1_카페쿠피입출고관리대장(관리자용YYYYMMDD)_X월마감.xlsx"
file_pattern = re.compile(r"(\d+)_카페쿠피입출고관리대장\(관리자용(\d{6,8})\)_\d+월마감\.xlsx")

# 🔹 폴더 내 모든 엑셀 파일 검색 및 정렬 (숫자순)
files = []
for f in os.listdir(data_dir):
    match = file_pattern.match(f)
    if match:
        files.append((int(match.group(1)), match.group(2), f))  # (숫자순, 날짜, 파일명) 저장
files.sort()  # 숫자 기준으로 정렬

# 🔹 품목명 → 품목_id 매핑 (공백 정리 포함)
품목_mapping = dict(zip(품목["품목명"].map(clean_text), 품목["품목_id"]))

# 🔹 매칭되지 않은 품목 저장용 리스트
매칭되지_않은_품목 = []

# 🔹 품목명 변경 규칙
품목명_변경 = {
    "립톤 복숭아파우더(907g) 가격인상": "립톤 복숭아파우더(907g)(24년인상)"
}

# 🔹 파일별 처리
for num, date_str, file in files:
    file_path = os.path.join(data_dir, file)
    
    # 🔹 날짜에서 연도와 월 추출
    year, month = date_str[:4], date_str[4:6]

    # 🔹 엑셀 파일 로드 (4번째 행을 컬럼명으로 사용)
    df = pd.read_excel(file_path, sheet_name="본사창고", header=3, dtype=str)

    # 🔹 컬럼명 공백 정리
    df.columns = [clean_text(col) for col in df.columns]

    # 🔹 모든 문자열 컬럼에 공백 정리 적용
    df = df.apply(lambda col: col.map(clean_text) if col.dtype == "O" else col)

    # 🔹 필요 컬럼 선택 (출고량 관련 칼럼만)
    출고_칼럼 = ["품목명", "1주차 출고량", "2주차 출고량", "3주차 출고량", "4주차 출고량", "5주차 출고량"]
    
    # 🔹 컬럼이 존재하는지 확인 (없으면 스킵)
    if not all(col in df.columns for col in 출고_칼럼):
        print(f"⚠️ {file}에서 필요한 컬럼이 누락됨. 스킵합니다.")
        continue

    출고_df = df[출고_칼럼].copy()

    # 🔹 데이터 타입 변환 (출고량이 빈 경우 0으로 처리)
    출고_df.fillna(0, inplace=True)
    for col in 출고_칼럼[1:]:  # "1주차 출고량" ~ "5주차 출고량"
        출고_df[col] = 출고_df[col].astype(int)

    # 🔹 기간 매핑 (1주차 → YYYY.MM.1, 2주차 → YYYY.MM.2 ...)
    기간_mapping = {f"{i}주차 출고량": f"{year}.{month}.{i}" for i in range(1, 6)}

    # 🔹 데이터 추가
    출고_data = []

    for _, row in 출고_df.iterrows():
        품목명 = clean_text(row["품목명"])  # 🔹 품목명 공백 정리 후 매칭

        # 🔹 품목명 변경 적용
        if 품목명 in 품목명_변경:
            품목명 = 품목명_변경[품목명]

        품목_id = 품목_mapping.get(품목명)  # 🔹 변경된 품목명으로 매칭

        if not 품목_id:  # 품목명이 품목 테이블에 없을 경우
            매칭되지_않은_품목.append((file, 품목명))  # 🔹 파일명과 함께 저장
            continue  # 매칭되지 않는 품목은 스킵

        for 주차, 기간 in 기간_mapping.items():
            창고_출고량 = row[주차]
            출고_data.append(["ST_102", 품목_id, 기간, 창고_출고량])

    # 🔹 창고_출고 테이블에 추가
    출고_df_final = pd.DataFrame(출고_data, columns=["매장_id", "품목_id", "기간", "창고_출고량"])
    창고_출고 = pd.concat([창고_출고, 출고_df_final], ignore_index=True)

창고_출고

,매장_id,품목_id,기간,창고_출고량
0,ST_102,IT_101,2023.12.1,0
1,ST_102,IT_101,2023.12.2,0
2,ST_102,IT_101,2023.12.3,0
3,ST_102,IT_101,2023.12.4,0
4,ST_102,IT_101,2023.12.5,0
...,...,...,...,...
4610,ST_102,IT_171,2025.01.1,0
4611,ST_102,IT_171,2025.01.2,3
4612,ST_102,IT_171,2025.01.3,0
4613,ST_102,IT_171,2025.01.4,0


## 창고_입고

In [21]:
# 🔹 창고_입고 테이블 초기화
창고_입고 = pd.DataFrame(columns=["매장_id", "품목_id", "기간", "창고_입고량"])

# 🔹 데이터가 저장된 폴더 경로
data_dir = "data/입출고관리대장/"

# 🔹 파일명 패턴: "1_카페쿠피입출고관리대장(관리자용YYYYMMDD)_X월마감.xlsx"
file_pattern = re.compile(r"(\d+)_카페쿠피입출고관리대장\(관리자용(\d{6,8})\)_\d+월마감\.xlsx")

# 🔹 폴더 내 모든 엑셀 파일 검색 및 정렬 (숫자순)
files = []
for f in os.listdir(data_dir):
    match = file_pattern.match(f)
    if match:
        files.append((int(match.group(1)), match.group(2), f))  # (숫자순, 날짜, 파일명) 저장
files.sort()  # 숫자 기준으로 정렬

# 🔹 품목명 → 품목_id 매핑 (공백 정리 포함)
품목_mapping = dict(zip(품목["품목명"].map(clean_text), 품목["품목_id"]))

# 🔹 매칭되지 않은 품목 저장용 리스트
매칭되지_않은_품목 = []

# 🔹 품목명 변경 규칙
품목명_변경 = {
    "립톤 복숭아파우더(907g) 가격인상": "립톤 복숭아파우더(907g)(24년인상)"
}

# 🔹 파일별 처리
for num, date_str, file in files:
    file_path = os.path.join(data_dir, file)
    
    # 🔹 날짜에서 연도와 월 추출
    year, month = date_str[:4], date_str[4:6]

    # 🔹 엑셀 파일 로드 (4번째 행을 컬럼명으로 사용)
    df = pd.read_excel(file_path, sheet_name="본사창고", header=3, dtype=str)

    # 🔹 컬럼명 공백 정리
    df.columns = [clean_text(col) for col in df.columns]

    # 🔹 모든 문자열 컬럼에 공백 정리 적용
    df = df.apply(lambda col: col.map(clean_text) if col.dtype == "O" else col)

    # 🔹 필요 컬럼 선택 (입고량 관련 칼럼만)
    입고_칼럼 = ["품목명", "1주차 입고", "2주차 입고", "3주차 입고", "4주차 입고", "5주차 입고"]
    
    # 🔹 컬럼이 존재하는지 확인 (없으면 스킵)
    if not all(col in df.columns for col in 입고_칼럼):
        print(f"⚠️ {file}에서 필요한 컬럼이 누락됨. 스킵합니다.")
        continue

    입고_df = df[입고_칼럼].copy()

    # 🔹 데이터 타입 변환 (입고량이 빈 경우 0으로 처리)
    입고_df.fillna(0, inplace=True)
    for col in 입고_칼럼[1:]:  # "1주차 입고" ~ "5주차 입고"
        입고_df[col] = 입고_df[col].astype(int)

    # 🔹 기간 매핑 (1주차 → YYYY.MM.1, 2주차 → YYYY.MM.2 ...)
    기간_mapping = {f"{i}주차 입고": f"{year}.{month}.{i}" for i in range(1, 6)}

    # 🔹 데이터 추가
    입고_data = []

    for _, row in 입고_df.iterrows():
        품목명 = clean_text(row["품목명"])  # 🔹 품목명도 공백 정리 후 매칭

        # 🔹 품목명 변경 적용
        if 품목명 in 품목명_변경:
            품목명 = 품목명_변경[품목명]

        품목_id = 품목_mapping.get(품목명)  # 🔹 변경된 품목명으로 매칭

        if not 품목_id:  # 품목명이 품목 테이블에 없을 경우
            매칭되지_않은_품목.append((file, 품목명))  # 🔹 파일명과 함께 저장
            continue  # 매칭되지 않는 품목은 스킵

        for 주차, 기간 in 기간_mapping.items():
            창고_입고량 = row[주차]
            입고_data.append(["ST_102", 품목_id, 기간, 창고_입고량])

    # 🔹 창고_입고 테이블에 추가
    입고_df_final = pd.DataFrame(입고_data, columns=["매장_id", "품목_id", "기간", "창고_입고량"])
    창고_입고 = pd.concat([창고_입고, 입고_df_final], ignore_index=True)

창고_입고

,매장_id,품목_id,기간,창고_입고량
0,ST_102,IT_101,2023.12.1,0
1,ST_102,IT_101,2023.12.2,0
2,ST_102,IT_101,2023.12.3,0
3,ST_102,IT_101,2023.12.4,0
4,ST_102,IT_101,2023.12.5,0
...,...,...,...,...
4610,ST_102,IT_171,2025.01.1,0
4611,ST_102,IT_171,2025.01.2,0
4612,ST_102,IT_171,2025.01.3,0
4613,ST_102,IT_171,2025.01.4,0


## 창고_재고

In [25]:
# 🔹 창고_재고 테이블 초기화
창고_재고 = pd.DataFrame(columns=["매장_id", "품목_id", "기간", "창고_재고량"])

# 🔹 데이터가 저장된 폴더 경로
data_dir = "data/입출고관리대장/"

# 🔹 파일명 패턴: "1_카페쿠피입출고관리대장(관리자용YYYYMMDD)_X월마감.xlsx"
file_pattern = re.compile(r"(\d+)_카페쿠피입출고관리대장\(관리자용(\d{6,8})\)_\d+월마감\.xlsx")

# 🔹 폴더 내 모든 엑셀 파일 검색 및 정렬 (숫자순)
files = []
for f in os.listdir(data_dir):
    match = file_pattern.match(f)
    if match:
        files.append((int(match.group(1)), match.group(2), f))  # (숫자순, 날짜, 파일명) 저장
files.sort()  # 숫자 기준으로 정렬

# 🔹 품목명 → 품목_id 매핑 (공백 정리 포함)
품목_mapping = dict(zip(품목["품목명"].map(clean_text), 품목["품목_id"]))

# 🔹 매칭되지 않은 품목 저장용 리스트
매칭되지_않은_품목 = []

# 🔹 품목명 변경 규칙
품목명_변경 = {
    "립톤 복숭아파우더(907g) 가격인상": "립톤 복숭아파우더(907g)(24년인상)"
}

# 🔹 파일별 처리
for num, date_str, file in files:
    file_path = os.path.join(data_dir, file)
    
    # 🔹 날짜에서 연도와 월 추출
    year, month = date_str[:4], date_str[4:6]

    # 🔹 해당 월의 마지막 날짜 계산 (30일 or 31일)
    month_days = 31 if month in ["01", "03", "05", "07", "08", "10", "12"] else 30

    # 🔹 엑셀 파일 로드 (4번째 행을 컬럼명으로 사용)
    df = pd.read_excel(file_path, sheet_name="본사창고", header=3, dtype=str)

    # 🔹 컬럼명 공백 정리
    df.columns = [clean_text(col) for col in df.columns]

    # 🔹 모든 문자열 컬럼에 공백 정리 적용
    df = df.apply(lambda col: col.map(clean_text) if col.dtype == "O" else col)

    # 🔹 필요 컬럼 선택 (현재고만 사용)
    재고_칼럼 = ["품목명", "현재고"]
    
    # 🔹 컬럼이 존재하는지 확인 (없으면 스킵)
    if not all(col in df.columns for col in 재고_칼럼):
        print(f"⚠️ {file}에서 필요한 컬럼이 누락됨. 스킵합니다.")
        continue

    재고_df = df[재고_칼럼].copy()

    # 🔹 데이터 타입 변환 (현재고가 빈 경우 0으로 처리)
    재고_df.fillna(0, inplace=True)
    재고_df["현재고"] = 재고_df["현재고"].astype(int)

    # 🔹 데이터 추가
    재고_data = []

    for _, row in 재고_df.iterrows():
        품목명 = clean_text(row["품목명"])  # 🔹 품목명도 공백 정리 후 매칭

        # 🔹 품목명 변경 적용
        if 품목명 in 품목명_변경:
            품목명 = 품목명_변경[품목명]

        품목_id = 품목_mapping.get(품목명)  # 🔹 변경된 품목명으로 매칭

        if not 품목_id:  # 품목명이 품목 테이블에 없을 경우
            매칭되지_않은_품목.append((file, 품목명))  # 🔹 파일명과 함께 저장
            continue  # 매칭되지 않는 품목은 스킵

        창고_재고량 = row["현재고"]

        # 🔹 해당 월의 모든 날짜에 동일한 창고_재고량 입력
        for day in range(1, month_days + 1):
            기간 = f"{year}.{month}.{str(day).zfill(2)}"  # YYYY.MM.DD 형식
            재고_data.append(["ST_102", 품목_id, 기간, 창고_재고량])  # 🔹 매장_id 추가

    # 🔹 창고_재고 테이블에 추가
    재고_df_final = pd.DataFrame(재고_data, columns=["매장_id", "품목_id", "기간", "창고_재고량"])
    창고_재고 = pd.concat([창고_재고, 재고_df_final], ignore_index=True)

창고_재고

,매장_id,품목_id,기간,창고_재고량
0,ST_102,IT_101,2023.12.01,5000
1,ST_102,IT_101,2023.12.02,5000
2,ST_102,IT_101,2023.12.03,5000
3,ST_102,IT_101,2023.12.04,5000
4,ST_102,IT_101,2023.12.05,5000
...,...,...,...,...
28277,ST_102,IT_171,2025.01.27,14
28278,ST_102,IT_171,2025.01.28,14
28279,ST_102,IT_171,2025.01.29,14
28280,ST_102,IT_171,2025.01.30,14


## 매장_재고

In [33]:
import os
import pandas as pd
import re

# 🔹 매장_재고 테이블 초기화
매장_재고 = pd.DataFrame(columns=["매장_id", "품목_id", "기간", "매장_재고량"])

# 🔹 데이터가 저장된 폴더 경로
data_dir = "data/입출고관리대장/"

# 🔹 파일명 패턴
file_pattern = re.compile(r"(\d+)_카페쿠피입출고관리대장\(관리자용(\d{6,8})\)_\d+월마감\.xlsx")

# 🔹 탐색할 매장 시트 목록 (clean_text 적용)
매장_시트목록 = [clean_text(x) for x in ["푸른솔", "의과대학", "중앙도서관", "학생회관", "예술디자인대", "선승관", "공학관", "멀티미디어관", "제2기숙사"]]

# 🔹 폴더 내 모든 엑셀 파일 검색 및 정렬 (숫자순)
files = []
for f in os.listdir(data_dir):
    match = file_pattern.match(f)
    if match:
        files.append((int(match.group(1)), match.group(2), f))  # (숫자순, 날짜, 파일명) 저장
files.sort()  # 숫자 기준으로 정렬

# 🔹 품목명 → 품목_id 매핑 (공백 정리 포함)
품목_mapping = dict(zip(품목["품목명"].map(clean_text), 품목["품목_id"]))

# 🔹 매장명 → 매장_id 매핑 (공백 정리 포함)
매장_mapping = dict(zip(매장["매장명"].map(clean_text), 매장["매장_id"]))

# 🔹 매칭되지 않은 품목 저장용 리스트
매칭되지_않은_품목 = []

# 🔹 품목명 변경 규칙
품목명_변경 = {
    "10/13홀더 (1도인쇄)": "10/13홀더",
    "12/16홀더(1도인쇄)": "12/16홀더"
}

# 🔹 파일별 처리
for num, date_str, file in files:
    file_path = os.path.join(data_dir, file)
    print(num)
    # 🔹 날짜에서 연도와 월 추출 (전월 재고이므로 한 달 빼줌)
    year, month = int(date_str[:4]), int(date_str[4:6])
    if month == 1:
        year -= 1
        month = 12
    else:
        month -= 1
    month = str(month).zfill(2)  # 두 자리 숫자로 변환

    # 🔹 해당 월의 마지막 날짜 계산 (30일 or 31일)
    month_days = 31 if month in ["01", "03", "05", "07", "08", "10", "12"] else 30

    # 🔹 매장별 시트 탐색
    for 매장명 in 매장_시트목록:
        try:
            # 🔹 엑셀 파일 로드 (매장별 시트)
            df = pd.read_excel(file_path, sheet_name=매장명, header=3, dtype=str)

            # 🔹 시트명 공백 정리
            매장명 = clean_text(매장명)

            # 🔹 컬럼명 공백 정리
            df.columns = [clean_text(col) for col in df.columns]

            # 🔹 모든 문자열 컬럼에 공백 정리 적용
            df = df.apply(lambda col: col.map(clean_text) if col.dtype == "O" else col)

            # 🔹 필요 컬럼 선택 (전월 재고만 사용)
            재고_칼럼 = ["품목명", "전월 재고"]

            # 🔹 컬럼이 존재하는지 확인 (없으면 스킵)
            if not all(col in df.columns for col in 재고_칼럼):
                print(f"⚠️ {file}의 {매장명} 시트에서 필요한 컬럼이 누락됨. 스킵합니다.")
                continue

            재고_df = df[재고_칼럼].copy()

            # 🔹 전월 재고 데이터 변환 (NaN → 0.0, 빈 값 → 0.0, 소수점 유지)
            재고_df["전월 재고"] = 재고_df["전월 재고"].replace("", "0").astype(float).fillna(0.0)

            # 🔹 데이터 추가
            재고_data = []

            for _, row in 재고_df.iterrows():
                품목명 = clean_text(row["품목명"])  # 🔹 품목명도 공백 정리 후 매칭

                # 🔹 품목명 변경 적용
                if 품목명 in 품목명_변경:
                    품목명 = 품목명_변경[품목명]

                품목_id = 품목_mapping.get(품목명)  # 🔹 변경된 품목명으로 매칭
                매장_id = 매장_mapping.get(매장명)

                if not 품목_id:  # 품목명이 품목 테이블에 없을 경우
                    매칭되지_않은_품목.append((file, 매장명, 품목명))  # 🔹 파일명, 시트명과 함께 저장
                    continue  # 매칭되지 않는 품목은 스킵

                매장_재고량 = row["전월 재고"]

                # 🔹 해당 월의 모든 날짜에 동일한 매장_재고량 입력 (소수점 포함)
                for day in range(1, month_days + 1):
                    기간 = f"{year}.{month}.{str(day).zfill(2)}"  # YYYY.MM.DD 형식
                    재고_data.append([매장_id, 품목_id, 기간, 매장_재고량])  # 🔹 매장_id 추가

            # 🔹 매장_재고 테이블에 추가 (빈 데이터프레임 예외 처리 추가)
            재고_df_final = pd.DataFrame(재고_data, columns=["매장_id", "품목_id", "기간", "매장_재고량"])

            # 🔹 매장_재고 테이블에 추가 (무조건 데이터 삽입)
            if not 재고_df_final.empty:
                매장_재고 = pd.concat([매장_재고, 재고_df_final], ignore_index=True)

        except Exception as e:
            print(f"⚠️ {file}의 {매장명} 시트 로딩 실패: {e}")
            continue

매장_재고

1


C:\Users\Sun\AppData\Local\Temp\ipykernel_3548\4051206607.py:113: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  매장_재고 = pd.concat([매장_재고, 재고_df_final], ignore_index=True)


2
3
4
5
6
7
8
9
10
11
12
13
14


,매장_id,품목_id,기간,매장_재고량
0,ST_103,IT_101,2023.11.01,0.0
1,ST_103,IT_101,2023.11.02,0.0
2,ST_103,IT_101,2023.11.03,0.0
3,ST_103,IT_101,2023.11.04,0.0
4,ST_103,IT_101,2023.11.05,0.0
...,...,...,...,...
253696,ST_111,IT_171,2024.12.27,1.0
253697,ST_111,IT_171,2024.12.28,1.0
253698,ST_111,IT_171,2024.12.29,1.0
253699,ST_111,IT_171,2024.12.30,1.0


## 매장_발주

In [29]:
# 🔹 매장_발주 테이블 초기화
매장_발주 = pd.DataFrame(columns=["매장_id", "품목_id", "기간", "매장_발주량"])

# 🔹 데이터가 저장된 폴더 경로
data_dir = "data/입출고관리대장/"

# 🔹 파일명 패턴
file_pattern = re.compile(r"(\d+)_카페쿠피입출고관리대장\(관리자용(\d{6,8})\)_\d+월마감\.xlsx")

# 🔹 탐색할 매장 시트 목록 (clean_text 적용)
매장_시트목록 = [clean_text(x) for x in ["푸른솔", "의과대학", "중앙도서관", "학생회관", "예술디자인대", "선승관", "공학관", "멀티미디어관", "제2기숙사"]]

# 🔹 폴더 내 모든 엑셀 파일 검색 및 정렬 (숫자순)
files = []
for f in os.listdir(data_dir):
    match = file_pattern.match(f)
    if match:
        files.append((int(match.group(1)), match.group(2), f))  # (숫자순, 날짜, 파일명) 저장
files.sort()  # 숫자 기준으로 정렬

# 🔹 품목명 → 품목_id 매핑 (공백 정리 포함)
품목_mapping = dict(zip(품목["품목명"].map(clean_text), 품목["품목_id"]))

# 🔹 매장명 → 매장_id 매핑 (공백 정리 포함)
매장_mapping = dict(zip(매장["매장명"].map(clean_text), 매장["매장_id"]))

# 🔹 매칭되지 않은 품목 저장용 리스트
매칭되지_않은_품목 = []

# 🔹 품목명 변경 규칙
품목명_변경 = {
    "10/13홀더 (1도인쇄)": "10/13홀더",
    "12/16홀더(1도인쇄)": "12/16홀더"
}

# 🔹 파일별 처리
for num, date_str, file in files:
    file_path = os.path.join(data_dir, file)
    print(num)
    # 🔹 날짜에서 연도와 월 추출 (파일의 달 그대로 사용)
    year, month = date_str[:4], date_str[4:6]

    # 🔹 매장별 시트 탐색
    for 매장명 in 매장_시트목록:
        try:
            # 🔹 엑셀 파일 로드 (매장별 시트)
            df = pd.read_excel(file_path, sheet_name=매장명, header=3, dtype=str)

            # 🔹 시트명 공백 정리
            매장명 = clean_text(매장명)

            # 🔹 컬럼명 공백 정리
            df.columns = [clean_text(col) for col in df.columns]

            # 🔹 모든 문자열 컬럼에 공백 정리 적용
            df = df.apply(lambda col: col.map(clean_text) if col.dtype == "O" else col)

            # 🔹 필요 컬럼 선택 (발주량 관련 칼럼만)
            발주_칼럼 = ["품목명", "1주차 발주", "2주차 발주", "3주차 발주", "4주차 발주", "5주차 발주"]

            # 🔹 컬럼이 존재하는지 확인 (없으면 스킵)
            if not all(col in df.columns for col in 발주_칼럼):
                print(f"⚠️ {file}의 {매장명} 시트에서 필요한 컬럼이 누락됨. 스킵합니다.")
                continue

            발주_df = df[발주_칼럼].copy()

            # 🔹 발주량 데이터 변환 (빈 값 → 0, 정수 변환)
            for col in 발주_칼럼[1:]:  # "1주차 발주" ~ "5주차 발주"
                발주_df[col] = pd.to_numeric(발주_df[col], errors="coerce").fillna(0).astype(int)

            # 🔹 기간 매핑 (1주차 → YYYY.MM.1, 2주차 → YYYY.MM.2 ...)
            기간_mapping = {f"{i}주차 발주": f"{year}.{month}.{i}" for i in range(1, 6)}

            # 🔹 데이터 추가
            발주_data = []

            for _, row in 발주_df.iterrows():
                품목명 = clean_text(row["품목명"])  # 🔹 품목명도 공백 정리 후 매칭

                # 🔹 품목명 변경 적용
                if 품목명 in 품목명_변경:
                    품목명 = 품목명_변경[품목명]

                품목_id = 품목_mapping.get(품목명)  # 🔹 변경된 품목명으로 매칭
                매장_id = 매장_mapping.get(매장명)

                if not 품목_id:  # 품목명이 품목 테이블에 없을 경우
                    매칭되지_않은_품목.append((file, 매장명, 품목명))  # 🔹 파일명, 시트명과 함께 저장
                    continue  # 매칭되지 않는 품목은 스킵

                for 주차, 기간 in 기간_mapping.items():
                    매장_발주량 = row[주차]
                    발주_data.append([매장_id, 품목_id, 기간, 매장_발주량])  # 🔹 매장_id 추가

            # 🔹 매장_발주 테이블에 추가 (빈 데이터프레임 예외 처리 추가)
            발주_df_final = pd.DataFrame(발주_data, columns=["매장_id", "품목_id", "기간", "매장_발주량"])

            # 🔹 데이터프레임이 비어있더라도 강제로 추가
            if not 발주_df_final.empty:
                매장_발주 = pd.concat([매장_발주, 발주_df_final], ignore_index=True)

        except Exception as e:
            print(f"⚠️ {file}의 {매장명} 시트 로딩 실패: {e}")
            continue

매장_발주

1
2
3
4
5
6
7
8
9
10
11
12
13
14


,매장_id,품목_id,기간,매장_발주량
0,ST_103,IT_101,2023.12.1,0
1,ST_103,IT_101,2023.12.2,0
2,ST_103,IT_101,2023.12.3,0
3,ST_103,IT_101,2023.12.4,0
4,ST_103,IT_101,2023.12.5,0
...,...,...,...,...
41485,ST_111,IT_171,2025.01.1,0
41486,ST_111,IT_171,2025.01.2,0
41487,ST_111,IT_171,2025.01.3,0
41488,ST_111,IT_171,2025.01.4,0


---

In [38]:
# 🔹 저장할 폴더 경로
save_dir = "data/출력데이터/"

# 🔹 폴더가 없으면 생성
os.makedirs(save_dir, exist_ok=True)

# 🔹 테이블 리스트
tables = {
    "매장": 매장,
    "협력사": 협력사,
    "품목": 품목,
    "매장_재고": 매장_재고,
    "매장_발주": 매장_발주,
    "창고_입고": 창고_입고,
    "창고_출고": 창고_출고,
    "창고_재고": 창고_재고,
}

# 🔹 CSV 파일로 저장
for name, df in tables.items():
    file_path = os.path.join(save_dir, f"{name}.csv")
    df.to_csv(file_path, index=False, encoding="utf-8-sig")  # 🔹 UTF-8 인코딩으로 저장

print("✅ 모든 테이블이 CSV 파일로 저장되었습니다!")

✅ 모든 테이블이 CSV 파일로 저장되었습니다!


---

In [53]:
import os
import pymysql
import pandas as pd

# 🔹 MySQL 연결 정보
DB_CONFIG = {
    "host": "localhost",  # MySQL 서버 주소
    "user": "root",       # MySQL 사용자 이름
    "password": "9420",   # MySQL 비밀번호
    "database": "khuffee",  # 사용할 데이터베이스명
    "charset": "utf8mb4"
}

# 🔹 CSV 파일이 저장된 폴더 경로
csv_dir = "data/출력데이터/"

# 🔹 테이블과 CSV 파일 매핑
TABLES = {
    "매장": "매장.csv",
    "협력사": "협력사.csv",
    "품목": "품목.csv",
    "매장_재고": "매장_재고.csv",
    "매장_발주": "매장_발주.csv",
    "창고_입고": "창고_입고.csv",
    "창고_출고": "창고_출고.csv",
    "창고_재고": "창고_재고.csv",
    # "창고_발주": "창고_발주.csv",
    # "매출": "매출.csv",
}

# 🔹 MySQL 연결
connection = pymysql.connect(**DB_CONFIG)
cursor = connection.cursor()

# 🔹 테이블별 INSERT SQL 문 (필드 순서 일치해야 함)
INSERT_QUERIES = {
    "매장": "INSERT INTO 매장 (매장_id, 매장명, 매장_비밀번호) VALUES (%s, %s, %s)",
    "협력사": "INSERT INTO 협력사 (협력사_id, 협력사명) VALUES (%s, %s)",
    "품목": "INSERT INTO 품목 (품목_id, 협력사_id, 품목명, 규격, 단위, 입고단가, 입고단위, 입고단위단가) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)",
    "매장_재고": "INSERT INTO 매장_재고 (매장_id, 품목_id, 기간, 매장_재고량) VALUES (%s, %s, %s, %s)",
    "매장_발주": "INSERT INTO 매장_발주 (매장_id, 품목_id, 기간, 매장_발주량) VALUES (%s, %s, %s, %s)",
    "창고_입고": "INSERT INTO 창고_입고 (매장_id, 품목_id, 기간, 창고_입고량) VALUES (%s, %s, %s, %s)",
    "창고_출고": "INSERT INTO 창고_출고 (매장_id, 품목_id, 기간, 창고_출고량) VALUES (%s, %s, %s, %s)",
    "창고_재고": "INSERT INTO 창고_재고 (매장_id, 품목_id, 기간, 창고_재고량) VALUES (%s, %s, %s, %s)",
    # "창고_발주": "INSERT INTO 창고_발주 (협력사_id, 품목_id, 기간, 창고_발주량) VALUES (%s, %s, %s, %s)",
    # "매출": "INSERT INTO 매출 (매장_id, 기간, 매출액) VALUES (%s, %s, %s)",
}

# 🔹 테이블 적재 실행
for table, file_name in TABLES.items():
    file_path = os.path.join(csv_dir, file_name)
    
    # CSV 파일이 존재하는지 확인
    if not os.path.exists(file_path):
        print(f"⚠️ 파일 {file_name}이(가) 존재하지 않습니다. 스킵합니다.")
        continue
    
    # CSV 데이터 읽기
    df = pd.read_csv(file_path, dtype=str)  # 모든 데이터를 문자열로 읽음

    # NULL 값 처리 (빈 값 → None)
    df = df.where(pd.notnull(df), None)

    # 🔹 float 변환이 필요한 컬럼 확인 후 변환
    if "입고단가" in df.columns:
        df["입고단가"] = df["입고단가"].astype(float)
    if "매장_재고량" in df.columns:
        df["매장_재고량"] = df["매장_재고량"].astype(float)
    if "매출액" in df.columns:
        df["매출액"] = df["매출액"].astype(float)

    # 데이터 삽입
    values = [tuple(row) for row in df.to_numpy()]
    
    try:
        cursor.executemany(INSERT_QUERIES[table], values)
        connection.commit()
        print(f"✅ {table} 테이블에 {len(df)}개 레코드 삽입 완료!")
    except Exception as e:
        connection.rollback()
        print(f"❌ {table} 테이블 데이터 삽입 오류: {e}")

# 🔹 연결 종료
cursor.close()
connection.close()
print("✅ 모든 데이터 삽입 완료!")

✅ 매장 테이블에 11개 레코드 삽입 완료!
✅ 협력사 테이블에 6개 레코드 삽입 완료!
✅ 품목 테이블에 71개 레코드 삽입 완료!
✅ 매장_재고 테이블에 253701개 레코드 삽입 완료!
✅ 매장_발주 테이블에 41490개 레코드 삽입 완료!
✅ 창고_입고 테이블에 4615개 레코드 삽입 완료!
✅ 창고_출고 테이블에 4615개 레코드 삽입 완료!
✅ 창고_재고 테이블에 28282개 레코드 삽입 완료!
✅ 모든 데이터 삽입 완료!
